# Chapter 4 — Structural VAR Identification
## Example 1: Cholesky Decomposition · Example 2: Proxy SVAR

**Macroeconometrics** | Alessia Paccagnini | Companion Notebook

---

Chapter 3 showed that reduced-form VARs are unidentified: the innovations $u_t$ are correlated mixtures of structural shocks, and IRFs cannot be given causal interpretations without further restrictions. Chapter 4 presents two leading identification strategies:

| | **Example 1 — Cholesky** | **Example 2 — Proxy SVAR** |
|---|---|---|
| **Strategy** | Recursive timing restrictions on $B_0$ | External instrument (narrative shocks) |
| **Assumption** | Strict causal ordering (lower-triangular $B_0$) | Instrument correlated with MP shock only |
| **Data** | FRED-QD (`2026-01-QD.xlsx`): GDP, Oil, CPI deflator, Fed funds | Individual FRED files + RR shock |
| **System** | 4-variable VAR(4): GDP growth, Oil, Inflation, Fed funds | 3-variable VAR(4): GDP growth, Inflation, Fed funds |
| **Sample** | 1971:Q1 – 2007:Q4 | 1970:Q1 – 2007:Q4 |
| **Output** | IRFs + 68/90% bootstrap bands, cumulative responses, FEVD, historical decomposition | Proxy SVAR IRFs (wild bootstrap), Cholesky comparison |

**Design conventions:**
- 🎛️ cells mark tunable parameters
- **Header comments** explain *why* each step is needed
- **Inline comments** explain *what* each line does
- **"What to look for"** notes follow every figure
- All figures saved as `.png` and `.pdf`

---

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.linalg import cholesky, inv
from scipy.ndimage import uniform_filter1d
import warnings, os
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = '.'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Shared colour palette (used in both examples) ────────────────────────────
COLORS = {
    'gdp':       '#2E86AB',
    'oil':       '#06A77D',
    'inflation': '#A23B72',
    'rate':      '#F18F01',
    'cholesky':  '#E94F37',
    'proxy':     '#2E86AB',
}

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.fontsize': 9,
})

print('All packages loaded.')

---
# Example 1 — Cholesky Identification
### 4-Variable SVAR: GDP Growth, Oil Prices, Inflation, Federal Funds Rate
*Textbook reference: Section 4.18.1*

## Background

The **Cholesky decomposition** (Sims, 1980) identifies a recursive SVAR by imposing a lower-triangular structure on the contemporaneous impact matrix $B_0$:

$$B_0 \, u_t = \varepsilon_t \quad \Leftrightarrow \quad u_t = B_0^{-1} \varepsilon_t = P \varepsilon_t$$

where $P = \text{chol}(\Sigma_u)$ is the lower-triangular Cholesky factor of the reduced-form covariance matrix. The ordering $(Y_t, Q_t, \pi_t, i_t)$ imposes:
- GDP growth $Y_t$: does not respond contemporaneously to **any** other shock (most predetermined)
- Oil price growth $Q_t$: responds to GDP shock, but not to inflation or Fed funds within the quarter
- Inflation $\pi_t$: responds to GDP and oil shocks, but not Fed funds within the quarter
- Fed funds rate $i_t$: responds to all other variables within the quarter (most endogenous)

This ordering — real variables before prices before policy — follows Christiano, Eichenbaum & Evans (1999). Adding oil prices follows Kim & Roubini (2000) and helps avoid the **price puzzle**: inflation appearing to rise after a contractionary monetary policy shock.

## Data: FRED-QD

| Variable | FRED-QD code | Transformation |
|---|---|---|
| Real GDP growth | `GDPC1` | $100 \times \Delta \ln$ |
| Oil price growth | `OILPRICEx` | $100 \times \Delta \ln$ |
| Inflation (GDP chain) | `GDPCTPI` | $100 \times \Delta \ln$ |
| Federal Funds Rate | `FEDFUNDS` | Level (%) |

In [ ]:
# 🎛️ EXAMPLE 1 PARAMETERS
PATH_FREDQD   = '2026-01-QD.xlsx'
E1_SAMPLE_START = '1971-01-01'
E1_SAMPLE_END   = '2007-12-31'
E1_P            = 4      # VAR lag order
E1_H            = 20     # IRF horizon (quarters)
E1_N_BOOT       = 300    # bootstrap replications (use 1000 for publication)

# Cholesky ordering: index 3 = Fed funds = monetary policy shock
E1_SHOCK_IDX   = 3
E1_VAR_NAMES   = ['GDP Growth', 'Oil Price Growth', 'Inflation', 'Federal Funds']
E1_SHOCK_COLORS= [COLORS['gdp'], COLORS['oil'], COLORS['inflation'], COLORS['rate']]

### Step 1 — Load and prepare FRED-QD data

The FRED-QD file has a specific structure: rows 0–1 are metadata, row 2 is the first data row (1959:Q3). We start from `iloc[2:]` to skip the two metadata rows, then select the columns we need by name.

In [ ]:
# ── Load FRED-QD ──────────────────────────────────────────────────────────────
# The first two rows after the header are metadata (factor codes, transform codes)
# Row index 2 is the first actual data row (1959:Q3 in the 2026-01 vintage)
df_raw = pd.read_excel(PATH_FREDQD)
df_raw = df_raw.iloc[2:].copy()   # skip the two metadata rows
df_raw['sasdate'] = pd.to_datetime(df_raw['sasdate'])
df_raw = df_raw.set_index('sasdate')

# Convert to numeric — FRED-QD can store some cells as strings
for var in ['GDPC1', 'GDPCTPI', 'FEDFUNDS', 'OILPRICEx']:
    df_raw[var] = pd.to_numeric(df_raw[var], errors='coerce')

# ── Transform to stationary series ───────────────────────────────────────────
# All growth rates: 100 × Δln = quarter-on-quarter percent change (NOT annualised)
# The textbook uses non-annualised QoQ rates for the 4-variable SVAR to keep the
# IRF units comparable across the variables (all in percentage points)
df = pd.DataFrame({
    'gdp_growth': 100 * np.log(df_raw['GDPC1']     / df_raw['GDPC1'].shift(1)),
    'oil_growth': 100 * np.log(df_raw['OILPRICEx'] / df_raw['OILPRICEx'].shift(1)),
    'inflation':  100 * np.log(df_raw['GDPCTPI']   / df_raw['GDPCTPI'].shift(1)),
    'fedfunds':   df_raw['FEDFUNDS']   # level in percent — already stationary
})

# Restrict to estimation sample and drop leading NaN from log-difference
df = df[E1_SAMPLE_START:E1_SAMPLE_END].dropna()

first_q = f"{df.index[0].year}:Q{(df.index[0].month-1)//3+1}"
last_q  = f"{df.index[-1].year}:Q{(df.index[-1].month-1)//3+1}"

print(f'Sample: {first_q} – {last_q}  |  T = {len(df)} observations')
print()
print(df.describe().round(3))

In [ ]:
# ── Figure E1-0: Data overview ────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)
fig.suptitle(f'Figure E1-0: 4-Variable SVAR Data  |  {first_q} – {last_q}\n'
             f'FRED-QD: GDPC1, OILPRICEx, GDPCTPI, FEDFUNDS',
             fontsize=12, fontweight='bold')

specs = list(zip(df.columns, E1_VAR_NAMES, E1_SHOCK_COLORS))
for ax, (col, title, color) in zip(axes, specs):
    ax.plot(df.index, df[col], color=color, linewidth=0.9)
    ax.axhline(0, color='black', linewidth=0.6, alpha=0.4)
    ax.set_title(title)
    ax.set_ylabel('%')

# NBER recessions
recessions = [('1973-10-01','1975-01-01'),('1980-01-01','1980-07-01'),
              ('1981-07-01','1982-10-01'),('1990-07-01','1991-01-01'),
              ('2001-01-01','2001-10-01')]
for ax in axes:
    for s, e in recessions:
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.10, color='grey')

axes[-1].set_xlabel('Quarter')
fig.tight_layout()
path = os.path.join(OUT_DIR, 'e1_fig0_data.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show(); print(f'Saved: {path}')

**What to look for:** The oil price series shows the two 1973 and 1979 oil shocks (large positive spikes) followed by the 1986 collapse (large negative spike). These are visible external supply shocks that motivate including oil prices in the SVAR — without them, the price puzzle is more severe because the VAR cannot distinguish cost-push inflation from demand-pull inflation driven by monetary loosening.

### Step 2 — VAR(4) estimation by OLS

In [ ]:
# ── OLS estimation of VAR(p) ─────────────────────────────────────────────────
# For a standard VAR, equation-by-equation OLS is equivalent to ML.
# We implement this manually (rather than using statsmodels) to have full
# control over the companion matrix and bootstrap, and to replicate the
# textbook's MATLAB results exactly.

def estimate_var_ols(Y, p):
    """
    Estimate VAR(p) by OLS.

    Y : T × K data matrix (numpy array)
    p : lag order

    Returns
    -------
    T_eff : effective sample size T - p
    B_hat : (1 + K*p) × K coefficient matrix  [const; A1; A2; ...; Ap]
    U     : T_eff × K residual matrix
    Sigma : K × K residual covariance matrix (degrees-of-freedom corrected)
    X     : T_eff × (1 + K*p) regressor matrix
    """
    T, K  = Y.shape
    T_eff = T - p
    Y_dep = Y[p:]    # T_eff × K dependent variable matrix

    # Build regressor matrix X = [ones | Y_{t-1} | ... | Y_{t-p}]
    X = np.ones((T_eff, 1))
    for lag in range(1, p + 1):
        X = np.hstack([X, Y[p - lag: T - lag]])

    # OLS: B_hat = (X'X)^{-1} X'Y_dep
    B_hat = np.linalg.lstsq(X, Y_dep, rcond=None)[0]
    U     = Y_dep - X @ B_hat    # T_eff × K residuals

    # Covariance: divide by T_eff - K*p - 1 (degrees-of-freedom correction)
    Sigma = (U.T @ U) / (T_eff - K * p - 1)
    return T_eff, B_hat, U, Sigma, X

Y  = df.values          # T × K data matrix
K  = Y.shape[1]
T  = Y.shape[0]

T_eff, B_hat, U, Sigma_u, X_var = estimate_var_ols(Y, E1_P)

print(f'VAR({E1_P}) estimated by OLS')
print(f'Effective sample: T_eff = {T_eff}  (T = {T} - p = {E1_P})')
print(f'Parameters per equation: {1 + K*E1_P}  (1 constant + {K} × {E1_P} lags)')
print(f'Total parameters: {K * (1 + K*E1_P)}')
print(f'\nReduced-form covariance matrix Σ̂ (×1000):')
print((1000*Sigma_u).round(2))

### Step 3 — Cholesky identification

The Cholesky factor $P = \text{chol}(\Sigma_u)$ satisfies $\Sigma_u = PP'$. It is the unique lower-triangular matrix with positive diagonal entries such that $P P' = \Sigma_u$. We set $B_0^{-1} = P$, so $u_t = P \varepsilon_t$ with $\text{Cov}(\varepsilon_t) = I_K$.

The **structural impact multiplier** $P_{ij}$ gives the contemporaneous response of variable $i$ to a unit structural shock $j$. The diagonal entry $P_{K,K} = P_{4,4}$ is the standard deviation of the monetary policy shock innovation (the size of a "unit" MP shock).

In [ ]:
# ── Cholesky decomposition ────────────────────────────────────────────────────
# cholesky(Sigma, lower=True) returns P such that P @ P.T = Sigma
# lower=True ensures P is LOWER triangular (standard convention in macroeconometrics)
P_chol = cholesky(Sigma_u, lower=True)

# The monetary policy shock has size P[3,3] = std dev of the MP innovation
mp_shock_size = P_chol[E1_SHOCK_IDX, E1_SHOCK_IDX]

print('Cholesky factor P  (lower-triangular impact matrix):')
print(pd.DataFrame(P_chol.round(4),
                   index=E1_VAR_NAMES,
                   columns=[f'ε_{n[:3]}' for n in E1_VAR_NAMES]))
print(f'\nMonetary policy shock size (P[3,3]): {mp_shock_size:.4f} pp')
print('Interpretation: a 1-standard-deviation MP shock raises the Fed funds rate '
      f'by {mp_shock_size:.2f} percentage points on impact.')
print()
print('Triangular structure (zeros above diagonal) = recursive restrictions:')
print('  Row 1 (GDP growth): responds to ε_GDP only — most predetermined')
print('  Row 4 (Fed funds) : responds to all shocks — most endogenous')

### Step 4 — IRF computation

The structural IRF at horizon $h$ is:
$$\Theta_h = \Phi_h P$$

where $\Phi_h$ is the $h$-th moving-average coefficient of the reduced-form VAR, computed from the companion matrix recursion $\Phi_h = \sum_{j=1}^{\min(h,p)} A_j \Phi_{h-j}$.

In [ ]:
# ── Build companion matrix and compute IRFs ───────────────────────────────────

def compute_irfs(B_hat, P, H, K, p):
    """
    Compute structural IRFs via the companion matrix.

    The companion matrix F stacks the VAR(p) into a VAR(1):
      [Y_t]         [A1 A2 ... Ap] [Y_{t-1}]
      [Y_{t-1}]  =  [I  0  ...  0] [Y_{t-2}] + ...
       ...

    The MA coefficients satisfy Φ_h = J F^h J', where J selects the top K rows.
    The structural IRF is Θ_h = Φ_h P.

    Returns IRF array of shape (H+1, K, K):
      IRF[h, i, j] = response of variable i at horizon h to structural shock j
    """
    # Build (Kp × Kp) companion matrix F
    F = np.zeros((K*p, K*p))
    # Top K rows: lag coefficient matrices A1, A2, ..., Ap
    # B_hat row 0 = constant; rows 1:K+1 = A1; rows K+1:2K+1 = A2; etc.
    F[:K, :] = B_hat[1:, :].T
    if p > 1:
        # Sub-diagonal identity block: shifts lags down by one period
        F[K:, :K*(p-1)] = np.eye(K*(p-1))

    # Selection matrix J: picks top K rows from companion vector
    J = np.hstack([np.eye(K), np.zeros((K, K*(p-1)))])

    IRF = np.zeros((H+1, K, K))
    IRF[0] = P         # impact effect at h=0: Θ_0 = P
    Fh = np.eye(K*p)   # F^0 = I
    for h in range(1, H+1):
        Fh = Fh @ F              # F^h = F^{h-1} × F
        IRF[h] = J @ Fh @ J.T @ P   # Θ_h = Φ_h P
    return IRF

IRF = compute_irfs(B_hat, P_chol, E1_H, K, E1_P)
# Extract the monetary policy shock column (index E1_SHOCK_IDX = 3)
irf_mp = IRF[:, :, E1_SHOCK_IDX]   # shape (H+1, K)

print(f'IRF shape: {IRF.shape}  (H+1 × K × K = {E1_H+1} × {K} × {K})')
print(f'\nImpact effects (h=0) of monetary policy shock:')
for i, name in enumerate(E1_VAR_NAMES):
    print(f'  {name:<20}: {irf_mp[0,i]:+.4f} pp')
print(f'\nNote: GDP growth and Oil have non-zero impact effects because they appear BEFORE\n'
      f'the Fed funds rate in the Cholesky order — the triangular structure does NOT\n'
      f'imply zero impact on earlier-ordered variables.')

### Step 5 — Bootstrap confidence bands

We use a **residual bootstrap** (Hall, 1985) with bias correction. The procedure:
1. Centre the residuals: $\tilde{u}_t = u_t - \bar{u}$
2. Draw bootstrap residuals $u_t^* = \tilde{u}_{\pi(t)}$ by resampling rows with replacement
3. Reconstruct $Y_t^*$ recursively from $\hat{B}$ and $u_t^*$
4. Re-estimate VAR and re-compute IRFs
5. Bias-correct: $\hat{\Theta}^{bc} = 2\hat{\Theta} - \bar{\Theta}^*$ where $\bar{\Theta}^*$ is the bootstrap mean

In [ ]:
# ── Residual bootstrap with bias correction ───────────────────────────────────
# This is the standard approach in the SVAR literature (Kilian 1998, Lütkepohl 2005)
# Bias correction accounts for the upward bias in autoregressive coefficient
# estimates in small samples.

print(f'Bootstrap: {E1_N_BOOT} replications...')

IRF_boot = np.zeros((E1_N_BOOT, E1_H+1, K, K))
U_ctr    = U - U.mean(axis=0)    # centre residuals for the resampling

for b in range(E1_N_BOOT):
    if (b+1) % 100 == 0:
        print(f'  {b+1}/{E1_N_BOOT}')

    # Draw bootstrap residuals by resampling rows of centred U
    idx    = np.random.randint(0, len(U_ctr), len(U_ctr))
    U_star = U_ctr[idx]

    # Reconstruct bootstrap data recursively:
    # Y_t* = B̂ × X_t* + u_t*,  initialised with actual pre-sample values
    Y_star = np.zeros((T, K))
    Y_star[:E1_P] = Y[:E1_P]    # use actual initial conditions
    for t in range(E1_P, T):
        # Build the regressor row for period t
        lags = Y_star[t-E1_P:t][::-1].flatten()   # lags 1..p stacked
        X_t  = np.concatenate([[1], lags])
        Y_star[t] = X_t @ B_hat + U_star[t - E1_P]

    # Re-estimate VAR on bootstrap data
    _, B_b, U_b, Sig_b, _ = estimate_var_ols(Y_star, E1_P)

    try:
        P_b = cholesky(Sig_b, lower=True)   # Cholesky of bootstrap Σ
        IRF_boot[b] = compute_irfs(B_b, P_b, E1_H, K, E1_P)
    except np.linalg.LinAlgError:
        # Cholesky fails if bootstrap Σ is not positive definite
        IRF_boot[b] = IRF    # fall back to point estimate

# Bias correction: bias = E*[IRF*] - IRF_point
bias      = IRF_boot.mean(axis=0) - IRF
# Bias-corrected bootstrap distribution (centred around point estimate)
IRF_bc    = IRF_boot - bias

# Percentile confidence bands from bias-corrected distribution
CI_68 = np.percentile(IRF_bc, [16, 84], axis=0)    # ±1 std equiv
CI_90 = np.percentile(IRF_bc, [5, 95],  axis=0)    # 90% CI

print(f'\nDone. CI_68 shape: {CI_68.shape}  (2 × H+1 × K × K)')

### Step 6 — FEVD and Historical Decomposition

**Forecast Error Variance Decomposition (FEVD):** At horizon $h$, the share of variable $i$'s forecast error variance attributable to shock $j$:

$$\text{FEVD}_{h,ij} = \frac{\sum_{s=0}^{h} \Theta_{s,ij}^2}{\sum_{s=0}^{h} [\Phi_s \Sigma_u \Phi_s']_{ii}}$$

**Historical Decomposition (HD):** The contribution of shock $j$ to the actual path of variable $i$ at time $t$:

$$\text{HD}_{t,ij} = \sum_{s=0}^{t} \Theta_{t-s,ij} \, \varepsilon_{s,j}$$

where $\varepsilon_{t,j} = [P^{-1} u_t]_j$ are the recovered structural shocks.

In [ ]:
# ── Forecast Error Variance Decomposition ────────────────────────────────────

def compute_fevd(IRF, H, K):
    """
    Compute FEVD from structural IRF array.

    FEVD[h, i, j] = fraction of variable i's h-step MSFE due to shock j.
    By construction: sum_j FEVD[h, i, j] = 1 for all h, i.
    """
    FEVD = np.zeros((H+1, K, K))
    for h in range(H+1):
        # MSE of variable i at horizon h = sum_{s=0}^{h} Σ_j Θ_{s,ij}^2
        mse = np.zeros((K, K))
        for s in range(h+1):
            mse += IRF[s] @ IRF[s].T    # K × K cumulative sum
        for i in range(K):
            total_var = mse[i, i]   # total forecast error variance of variable i
            for j in range(K):
                # Contribution of shock j to variable i's MSE
                shock_j_contrib = sum(IRF[s][i, j]**2 for s in range(h+1))
                FEVD[h, i, j] = shock_j_contrib / total_var if total_var > 0 else 0
    return FEVD

FEVD = compute_fevd(IRF, E1_H, K)

# Print FEVD at key horizons for the Fed funds equation
print('FEVD for Fed Funds Rate (% variance explained by each shock):')
print(f'{"Horizon":<10}  ' + '  '.join(f'{n[:7]:>8}' for n in E1_VAR_NAMES))
for h in [0, 4, 8, 12, 20]:
    row = FEVD[h, 3, :]   # row for Fed funds
    print(f'{h:<10}  ' + '  '.join(f'{100*v:>7.1f}%' for v in row))

print('\nFEVD for Inflation (% variance explained by each shock):')
print(f'{"Horizon":<10}  ' + '  '.join(f'{n[:7]:>8}' for n in E1_VAR_NAMES))
for h in [0, 4, 8, 12, 20]:
    row = FEVD[h, 2, :]   # row for inflation
    print(f'{h:<10}  ' + '  '.join(f'{100*v:>7.1f}%' for v in row))

In [ ]:
# ── Historical Decomposition ──────────────────────────────────────────────────

def historical_decomposition(Y, B_hat, P, p):
    """
    Compute historical decomposition (HD) of each variable into shock contributions.

    HD[t, i, j] = contribution of structural shock j to variable i at time t,
                  measured as deviations from the unconditional mean.

    Algorithm:
      1. Recover structural shocks: ε_t = P^{-1} u_t
      2. For each period t, compute: HD[t, :, j] = Σ_{s=0}^{t} Θ_{t-s} e_j ε_{s,j}
    """
    T_full, K = Y.shape

    # Rebuild regressor matrix for the full sample
    Y_lagged = [Y[p-lag:T_full-lag] for lag in range(1, p+1)]
    X_all    = np.column_stack([np.ones(T_full-p)] + Y_lagged)

    # Recover VAR residuals u_t
    U_all = Y[p:] - X_all @ B_hat

    # Recover structural shocks: ε_t = P^{-1} u_t
    eps = U_all @ inv(P).T    # (T-p) × K matrix of structural shocks

    # Build companion matrix (same as in compute_irfs)
    F = np.zeros((K*p, K*p))
    F[:K, :] = B_hat[1:, :].T
    if p > 1:
        F[K:, :K*(p-1)] = np.eye(K*(p-1))
    J = np.hstack([np.eye(K), np.zeros((K, K*(p-1)))])

    T_eff = T_full - p
    HD = np.zeros((T_eff, K, K))

    for t in range(T_eff):
        for shock in range(K):
            contrib = np.zeros(K)
            for s in range(t+1):
                # Θ_{t-s} = J F^{t-s} J' P  (structural MA coefficient)
                Fts    = np.linalg.matrix_power(F, t-s)
                impact = J @ Fts @ J.T @ P[:, shock]   # K-vector
                contrib += impact * eps[s, shock]       # scale by realised shock
            HD[t, :, shock] = contrib
    return HD, eps

HD, eps = historical_decomposition(Y, B_hat, P_chol, E1_P)

# Verify: sum of HD across shocks should recover the demeaned actual series
hd_sum  = HD.sum(axis=2)      # sum over shocks for each variable
actual  = Y[E1_P:] - Y[E1_P:].mean(axis=0)   # demeaned actual
max_err = np.abs(hd_sum - actual).max()
print(f'HD verification — max deviation from actual: {max_err:.6f}  '
      f'(should be near machine precision if implementation is correct)')

### Step 7 — Figures

In [ ]:
# ── Figure E1-1: IRFs to monetary policy shock ────────────────────────────────
# 4-panel plot: one panel per variable; two confidence bands (68% darker, 90% lighter)
# The dual-band convention follows Antolin-Diaz & Rubio-Ramirez (2018)
h_axis = np.arange(E1_H + 1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
fig.suptitle(f'Figure E1-1: IRFs to Monetary Policy Shock (Cholesky Identification)\n'
             f'4-Variable SVAR(4)  |  {first_q} – {last_q}  |  '
             f'Bootstrap: {E1_N_BOOT} reps',
             fontsize=12, fontweight='bold')

for i, (ax, name, color) in enumerate(zip(axes, E1_VAR_NAMES, E1_SHOCK_COLORS)):
    ax.plot(h_axis, irf_mp[:, i], color=color, linewidth=2.5, label='IRF', zorder=3)
    # 68% band (darker): roughly ±1 posterior std deviation
    ax.fill_between(h_axis, CI_68[0,:,i,E1_SHOCK_IDX], CI_68[1,:,i,E1_SHOCK_IDX],
                    alpha=0.35, color=color, label='68% CI', zorder=2)
    # 90% band (lighter): wider uncertainty region
    ax.fill_between(h_axis, CI_90[0,:,i,E1_SHOCK_IDX], CI_90[1,:,i,E1_SHOCK_IDX],
                    alpha=0.15, color=color, label='90% CI', zorder=1)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Quarters after shock')
    ax.set_title(name)
    if i == 0:
        ax.set_ylabel('Percentage Points')
    if i == 3:
        ax.legend(fontsize=8, loc='upper right')

fig.tight_layout()
path = os.path.join(OUT_DIR, 'e1_fig1_irfs.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show(); print(f'Saved: {path}')

**What to look for:** 
- **GDP growth** (panel 1): should eventually turn negative — a contractionary monetary policy shock reduces output. The effect typically peaks around $h=4$–$6$ quarters.
- **Oil price growth** (panel 2): should be small and close to zero — the monetary policy shock should not drive oil prices systematically (oil is included to absorb cost-push supply shocks, not as a transmission channel).
- **Inflation** (panel 3): should eventually turn negative. If it stays positive for the first few quarters, this is the **price puzzle** — residual endogeneity that the oil variable has not fully absorbed.
- **Fed funds rate** (panel 4): should show a positive impact effect (the shock itself) that decays toward zero — reflecting monetary policy persistence.

In [ ]:
# ── Figure E1-2: Cumulative responses (GDP level and price level) ─────────────
# Cumulating the growth-rate IRF recovers the response of the LOG LEVEL.
# This is economically more interpretable: how much does the price LEVEL fall
# following a monetary tightening?
irf_cum    = np.cumsum(irf_mp, axis=0)   # shape (H+1, K)
CI_68_cum  = np.cumsum(CI_68, axis=1)    # cumulate bands along horizon axis

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle(f'Figure E1-2: Cumulative Responses to Monetary Policy Shock\n'
             f'(= Level Response of Log GDP and Log Price Level)',
             fontsize=12, fontweight='bold')

for ax, var_idx, title, color in [
    (axes[0], 0, 'GDP Level Response',   COLORS['gdp']),
    (axes[1], 2, 'Price Level Response', COLORS['inflation'])
]:
    ax.fill_between(h_axis,
                    CI_68_cum[0,:,var_idx, E1_SHOCK_IDX],
                    CI_68_cum[1,:,var_idx, E1_SHOCK_IDX],
                    alpha=0.3, color=color)
    ax.plot(h_axis, irf_cum[:, var_idx], color=color, linewidth=2.5)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Quarters after shock')
    ax.set_ylabel('Percentage Points')
    ax.set_title(title)

fig.tight_layout()
path = os.path.join(OUT_DIR, 'e1_fig2_cumulative.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show(); print(f'Saved: {path}')

In [ ]:
# ── Figure E1-3: FEVD ─────────────────────────────────────────────────────────
# Stacked area chart: each color = one shock's contribution to the total
# forecast error variance. By construction the areas sum to 100% at every horizon.
shock_names  = ['GDP Shock', 'Oil Shock', 'Inflation Shock', 'MP Shock']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
fig.suptitle(f'Figure E1-3: Forecast Error Variance Decomposition\n'
             f'4-Variable Cholesky SVAR(4)  |  {first_q} – {last_q}',
             fontsize=13, fontweight='bold')

for i, (ax, var_name) in enumerate(zip(axes, E1_VAR_NAMES)):
    bottom = np.zeros(E1_H + 1)
    for j, (shock_name, shock_color) in enumerate(zip(shock_names, E1_SHOCK_COLORS)):
        # FEVD[:, i, j] = contribution of shock j to variable i, all horizons
        contrib = FEVD[:, i, j] * 100   # convert to percent
        ax.fill_between(h_axis, bottom, bottom + contrib,
                        label=shock_name, color=shock_color, alpha=0.82)
        bottom += contrib
    ax.set_xlabel('Quarters')
    ax.set_ylabel('Percent (%)')
    ax.set_title(var_name)
    ax.set_ylim(0, 100)
    ax.legend(loc='best', fontsize=9)

fig.tight_layout()
path = os.path.join(OUT_DIR, 'e1_fig3_fevd.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show(); print(f'Saved: {path}')

**What to look for in FEVD:** At $h=0$, the Cholesky ordering means the FEVD is exactly lower-triangular: 100% of GDP growth's variance is attributed to the GDP shock; the Fed funds rate variance at $h=0$ is split across all four shocks. As the horizon grows, cross-variable contributions increase. Key questions: (1) How much of inflation variation at long horizons ($h=20$) is explained by the MP shock? (2) How important are oil shocks for inflation vs GDP growth?

In [ ]:
# ── Figure E1-4: Historical Decomposition ────────────────────────────────────
# Stacked bar chart of shock contributions to each variable's demeaned path.
# The black line = actual demeaned series. The coloured areas should sum to it.
dates_hd = df.index[E1_P:]    # date index for the HD period

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
fig.suptitle(f'Figure E1-4: Historical Decomposition\n'
             f'4-Variable Cholesky SVAR(4)  |  {first_q} – {last_q}',
             fontsize=13, fontweight='bold')

for i, (ax, var_name) in enumerate(zip(axes, E1_VAR_NAMES)):
    bottom_pos = np.zeros(len(dates_hd))
    bottom_neg = np.zeros(len(dates_hd))

    for j, (shock_name, shock_color) in enumerate(zip(shock_names, E1_SHOCK_COLORS)):
        values   = HD[:, i, j]
        pos_vals = np.maximum(values, 0)   # positive contributions go above zero
        neg_vals = np.minimum(values, 0)   # negative contributions go below zero

        ax.fill_between(dates_hd, bottom_pos, bottom_pos + pos_vals,
                        label=shock_name, color=shock_color, alpha=0.72)
        ax.fill_between(dates_hd, bottom_neg, bottom_neg + neg_vals,
                        color=shock_color, alpha=0.72)   # no label (same shock)
        bottom_pos += pos_vals
        bottom_neg += neg_vals

    # Actual demeaned series (should equal sum of HD contributions)
    actual_dm = Y[E1_P:, i] - Y[E1_P:, i].mean()
    ax.plot(dates_hd, actual_dm, 'k-', linewidth=1.5, label='Actual (demeaned)', zorder=10)
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.4)

    ax.set_ylabel('Percentage Points')
    ax.set_title(var_name)
    ax.legend(loc='best', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', rotation=45, labelsize=8)

fig.tight_layout()
path = os.path.join(OUT_DIR, 'e1_fig4_historical_decomp.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show(); print(f'Saved: {path}')

**What to look for in HD:** Look for the oil shock contribution (green) to inflation in the 1973–1975 and 1979–1980 episodes — it should be large and positive, confirming that cost-push oil shocks drove the Great Inflation. The MP shock contribution (orange) to inflation should be negative during the Volcker disinflation (1981–1982), capturing the disinflationary effect of the large Fed funds rate increase.

---
# Example 2 — Proxy SVAR (External Instruments)
### 3-Variable SVAR with Romer-Romer Narrative Shocks
*Textbook reference: Section 4.18.11*

## Background

**The problem with Cholesky:** The recursive ordering imposes that GDP and inflation do not respond to monetary policy within the quarter. This is a strong assumption, and violations bias the IRFs.

**The Proxy SVAR solution** (Stock 2008, Mertens & Ravn 2013, Stock & Watson 2018): instead of restricting $B_0$, we use an **external instrument** $z_t$ that is correlated with the monetary policy shock $\varepsilon_t^{mp}$ but uncorrelated with all other structural shocks:

$$\mathbb{E}[z_t \varepsilon_t^{mp}] \neq 0 \quad \text{(relevance)}, \qquad \mathbb{E}[z_t \varepsilon_t^{j}] = 0 \; \forall j \neq mp \quad \text{(exogeneity)}$$

The identification strategy exploits:
$$\frac{\text{Cov}(u_t^j, z_t)}{\text{Cov}(u_t^{mp}, z_t)} = \frac{b_j}{b_{mp}}$$

where $b_j$ is the $j$-th element of the structural impact vector. Normalising $b_{mp} = 1$ gives us $b_j = \text{Cov}(u_t^j, z_t) / \text{Cov}(u_t^{mp}, z_t)$ — this is the core identification equation.

**Instrument:** Romer & Romer (2004) narrative shocks, updated by Wieland & Yang (2020). These are derived from FOMC meeting minutes and Fed forecasts: they measure the *exogenous* component of Fed rate decisions, purged of the Fed's endogenous reaction to the economy.

**Weak instrument test:** The first-stage $F$-statistic should exceed 10 (Stock & Yogo, 2005). Below 10, the proxy is too weakly correlated with $u_t^{mp}$ to identify the shock reliably.

In [ ]:
# 🎛️ EXAMPLE 2 PARAMETERS
PATH_GDP      = 'GDPC1.xlsx'
PATH_DEFL     = 'GDPDEF.xlsx'
PATH_FF       = 'FEDFUNDS.xlsx'
PATH_RR       = 'RR_monetary_shock_quarterly.xlsx'
E2_SAMPLE_START = '1970-01-01'
E2_SAMPLE_END   = '2007-12-31'
E2_P            = 4       # VAR lag order
E2_H            = 40      # IRF horizon (10 years)
E2_N_BOOT_PROXY = 2000    # wild bootstrap reps for proxy SVAR (Mammen weights)
E2_N_BOOT_CHOL  = 500     # residual bootstrap reps for Cholesky comparison
E2_CI_LEVEL     = 0.90    # confidence level

E2_VAR_NAMES  = ['GDP Growth', 'Inflation', 'Federal Funds Rate']

print(f'Example 2 parameters loaded.')
print(f'Bootstrap: {E2_N_BOOT_PROXY} proxy + {E2_N_BOOT_CHOL} Cholesky reps')
print(f'IRF horizon: {E2_H} quarters  |  CI level: {int(E2_CI_LEVEL*100)}%')

In [ ]:
# ── Load individual FRED files ────────────────────────────────────────────────
# The three FRED files have the same structure: README sheet + data sheet

def load_fred_quarterly(path, name):
    """Load quarterly FRED Excel file; return date-indexed DataFrame."""
    d = pd.read_excel(path, sheet_name='Quarterly', header=0)
    d.columns = ['date', name]
    d['date'] = pd.to_datetime(d['date']).dt.to_period('Q').dt.to_timestamp('Q')
    return d.set_index('date')

gdp_df  = load_fred_quarterly(PATH_GDP,  'GDPC1')
defl_df = load_fred_quarterly(PATH_DEFL, 'GDPDEF')

# Fed funds: monthly → quarterly average
ff_raw       = pd.read_excel(PATH_FF, sheet_name='Monthly', header=0)
ff_raw.columns = ['date', 'FEDFUNDS']
ff_raw['date'] = pd.to_datetime(ff_raw['date'])
ff_q         = ff_raw.set_index('date').resample('QE').mean()

# ── Compute 400 × Δln growth rates ───────────────────────────────────────────
# Use annualised rates here (×400) for consistency with proxy SVAR literature
gdp_growth = 400 * np.log(gdp_df['GDPC1']   / gdp_df['GDPC1'].shift(1))
inflation  = 400 * np.log(defl_df['GDPDEF'] / defl_df['GDPDEF'].shift(1))

# Normalise both to quarter-end dates before merging
gdp_growth.index = gdp_growth.index.to_period('Q').to_timestamp('Q')
inflation.index  = inflation.index.to_period('Q').to_timestamp('Q')

# ── Load Romer-Romer shocks ───────────────────────────────────────────────────
# Column 'resid_romer' = Romer-Romer (2004) residual, updated by Wieland-Yang (2020)
rr = pd.read_excel(PATH_RR)
rr['date'] = pd.to_datetime(rr['date']) + pd.offsets.QuarterEnd(0)  # align to QE
rr = rr.set_index('date')

print(f'RR shocks available: {rr.index[0].date()} – {rr.index[-1].date()}')
print(f'RR shock std dev: {rr["resid_romer"].std():.4f} pp')
print(f'RR shock mean:    {rr["resid_romer"].mean():.4f} pp  (should be ≈ 0)')

# ── Merge all series ──────────────────────────────────────────────────────────
data2 = pd.DataFrame({
    'gdp_growth': gdp_growth,
    'inflation':  inflation,
    'fedfunds':   ff_q['FEDFUNDS'],
    'rr_shock':   rr['resid_romer']
}).dropna()

data2 = data2.loc[E2_SAMPLE_START:E2_SAMPLE_END]
e2_first_q = f"{data2.index[0].year}:Q{data2.index[0].quarter}"
e2_last_q  = f"{data2.index[-1].year}:Q{data2.index[-1].quarter}"

print(f'\nProxy SVAR sample: {e2_first_q} – {e2_last_q}  (T = {len(data2)})')
print(data2[['gdp_growth','inflation','fedfunds']].describe().round(3))

In [ ]:
# ── Figure E2-0: Romer-Romer shock series ────────────────────────────────────
# Always inspect the instrument before using it — narrative shocks should be
# episodic (concentrated around major policy decisions) and mean-zero.
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle(f'Figure E2-0: Data and Instrument for Proxy SVAR  |  {e2_first_q} – {e2_last_q}',
             fontsize=12, fontweight='bold')

# Top panel: Fed funds rate (endogenous variable)
axes[0].plot(data2.index, data2['fedfunds'], color=COLORS['rate'], linewidth=1.0)
axes[0].set_title('Federal Funds Rate (endogenous variable)')
axes[0].set_ylabel('%')

# Bottom panel: Romer-Romer shock (the external instrument)
axes[1].bar(data2.index, data2['rr_shock'],
            color=np.where(data2['rr_shock'] >= 0, '#E74C3C', '#2E86AB'),
            width=70, alpha=0.75)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Romer-Romer (2004) Narrative Monetary Policy Shock (instrument $z_t$)\n'
                  'Red = contractionary, Blue = expansionary')
axes[1].set_ylabel('pp')
axes[1].set_xlabel('Quarter')

for ax in axes:
    for s, e in recessions:
        if pd.Timestamp(s) >= data2.index[0]:
            ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.10, color='grey')

fig.tight_layout()
path = os.path.join(OUT_DIR, 'e2_fig0_instrument.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show(); print(f'Saved: {path}')

**What to look for:** The Romer-Romer shock is concentrated around major policy turning points: large positive values in 1979–1980 (Volcker tightening) and large negative values in 1974–1975 and 2001 (easing episodes). Crucially, the shock should have **no** strong autocorrelation — a serially correlated instrument would suggest it is picking up endogenous Fed reactions rather than exogenous policy changes. The bar chart makes the episodic nature of narrative shocks visible.

### Step 2 — VAR estimation

In [ ]:
# ── Estimate VAR(4) for the 3-variable system ─────────────────────────────────
Y2     = data2[['gdp_growth', 'inflation', 'fedfunds']].values
K2     = Y2.shape[1]

T_eff2, B_hat2, U2, Sigma_u2, X_var2 = estimate_var_ols(Y2, E2_P)

# Align instrument with VAR residuals: instrument starts at period p+1
z = data2['rr_shock'].values[E2_P:]   # aligned to the residual period

print(f'VAR({E2_P}) estimated: T_eff = {T_eff2}')
print(f'Residual shape:    U2.shape = {U2.shape}')
print(f'Instrument length: len(z) = {len(z)}')
print(f'Instrument non-NaN: {np.sum(~np.isnan(z))}')

### Step 3 — Proxy SVAR identification

In [ ]:
# ── Proxy SVAR identification via covariance ratios ───────────────────────────

def proxy_svar_id(u, z):
    """
    Identify the structural impact vector for the MP shock using an
    external instrument z_t.

    Identification equation:
      b_j / b_mp = Cov(u_j, z) / Cov(u_mp, z)

    Normalise b_mp = 1, so:
      b_j = Cov(u_j, z) / Cov(u_mp, z)  for j = 1, ..., K

    This is the IV estimator applied to each equation:
      u_j = (b_j / b_mp) u_mp + noise  →  instrument u_mp with z.

    Parameters
    ----------
    u  : (T_eff × K) residual matrix from reduced-form VAR
    z  : (T_eff,) external instrument array

    Returns
    -------
    b       : K-vector of normalised structural impact multipliers
    se      : K-vector of bootstrap standard errors
    F_stat  : first-stage F-statistic (weak instrument test)
    R2      : first-stage R-squared
    n_valid : effective number of non-NaN instrument observations
    """
    K = u.shape[1]
    # Align lengths (take min)
    n = min(len(u), len(z))
    u, z = u[:n], z[:n]
    valid = ~np.isnan(z)
    u, z  = u[valid], z[valid]
    n_valid = len(z)

    # Core formula: b_j = Cov(u_j, z) / Cov(u_mp, z)
    cov_uz   = np.array([np.cov(u[:, j], z)[0, 1] for j in range(K)])
    cov_mp_z = cov_uz[-1]   # Cov(u_mp, z) — denominator
    b        = cov_uz / cov_mp_z   # normalised impact vector

    # First-stage regression: u_mp = α + β z + ε  (tests instrument relevance)
    X_fs     = np.column_stack([np.ones(n_valid), z])
    beta_fs  = np.linalg.lstsq(X_fs, u[:, -1], rcond=None)[0]
    resid_fs = u[:, -1] - X_fs @ beta_fs
    TSS  = np.sum((u[:, -1] - u[:, -1].mean())**2)
    RSS  = np.sum(resid_fs**2)
    R2   = 1 - RSS / TSS
    # F-stat with 1 instrument, n_valid observations
    F_stat = (R2 / 1) / ((1 - R2) / (n_valid - 2))

    # Bootstrap standard errors (500 reps, resampling rows)
    n_boot = 500
    b_boot = np.zeros((n_boot, K))
    for i in range(n_boot):
        idx = np.random.choice(n_valid, n_valid, replace=True)
        cov_b = np.array([np.cov(u[idx, j], z[idx])[0, 1] for j in range(K)])
        if np.abs(cov_b[-1]) > 1e-10:
            b_boot[i] = cov_b / cov_b[-1]
        else:
            b_boot[i] = np.nan
    se = np.nanstd(b_boot, axis=0)

    return b, se, F_stat, R2, n_valid

b_proxy, se_proxy, F_stat, R2, n_valid = proxy_svar_id(U2, z)

print('First-stage diagnostics (instrument relevance):')
print(f'  F-statistic = {F_stat:.1f}  '
      f'(Rule of thumb: F > 10 = strong instrument)')
print(f'  R-squared   = {R2:.4f}')
print(f'  Observations = {n_valid}')
print(f'  Verdict: {"✓ Strong instrument" if F_stat > 10 else "⚠ Weak instrument — results may be unreliable"}')

print('\nStructural impact multipliers b_j (normalised b_mp = 1):')
print(f'{"Variable":<25}  {"b_j":>10}  {"SE":>10}  {"t-stat":>10}')
print('-' * 60)
for j, name in enumerate(E2_VAR_NAMES):
    t = b_proxy[j] / se_proxy[j] if se_proxy[j] > 0 else np.nan
    print(f'{name:<25}  {b_proxy[j]:>10.4f}  {se_proxy[j]:>10.4f}  {t:>10.2f}')

**Interpreting the impact multipliers:** `b_j` measures the contemporaneous response of variable $j$ to a monetary policy shock, normalised so the Fed funds rate responds by 1 unit. Under the proxy approach, GDP growth and inflation are *allowed* to respond contemporaneously — there is no recursive zero restriction. A positive `b_gdp` means a contractionary shock (which raises the Fed funds rate) is associated with higher GDP growth on impact — this may seem paradoxical, but reflects endogeneity: periods when the Fed tightened often coincided with strong growth. The key economic content is at longer horizons.

### Step 4 — IRF computation (MA coefficients + proxy identification)

In [ ]:
# ── Moving-average coefficients and proxy IRFs ────────────────────────────────

def compute_ma_coefficients(B_hat, p, K, H):
    """
    Compute the reduced-form MA coefficients Φ_0, Φ_1, ..., Φ_H
    via the recursive relation Φ_h = Σ_{j=1}^{min(h,p)} A_j Φ_{h-j}.

    Returns Phi: (H+1, K, K) array with Phi[0] = I_K.
    """
    # Reshape B_hat into lag matrices A_1, ..., A_p
    # B_hat row 0 = constant; rows 1:K+1 = A_1; etc.
    B = B_hat[1:].reshape(p, K, K).transpose(0, 2, 1)
    # B[j] = A_{j+1}  (j=0,...,p-1)

    Phi    = np.zeros((H+1, K, K))
    Phi[0] = np.eye(K)   # Φ_0 = I_K by definition
    for h in range(1, H+1):
        for j in range(min(h, p)):
            Phi[h] += B[j] @ Phi[h - j - 1]   # Φ_h = Σ A_{j+1} Φ_{h-j-1}
    return Phi

def compute_irfs_proxy(B_hat, b, p, K, H):
    """
    Compute proxy SVAR IRFs.

    Unlike Cholesky which identifies all K shocks simultaneously,
    the proxy approach identifies ONE shock: the structural impact
    vector b is the column of B_0^{-1} corresponding to the MP shock.

    IRF_h = Φ_h b

    Returns irf: (H+1, K) array — one column per variable response.
    """
    Phi = compute_ma_coefficients(B_hat, p, K, H)
    irf = np.zeros((H+1, K))
    for h in range(H+1):
        irf[h] = Phi[h] @ b   # K-vector of responses at horizon h
    return irf

# Point estimate proxy IRFs
irf_proxy = compute_irfs_proxy(B_hat2, b_proxy, E2_P, K2, E2_H)

# Cholesky IRFs (for comparison in Figure E2-2)
P_chol2   = cholesky(Sigma_u2, lower=True)
def compute_irfs_chol3(B_hat, P, p, K, H):
    Phi = compute_ma_coefficients(B_hat, p, K, H)
    IRF = np.zeros((H+1, K, K))
    for h in range(H+1):
        IRF[h] = Phi[h] @ P
    return IRF
IRF_chol2  = compute_irfs_chol3(B_hat2, P_chol2, E2_P, K2, E2_H)
irf_chol_mp= IRF_chol2[:, :, 2]   # MP shock is last in Cholesky ordering

print(f'Proxy IRF shape: {irf_proxy.shape}  (H+1 × K)')
print(f'\nImpact effects (h=0) — proxy SVAR:')
for j, name in enumerate(E2_VAR_NAMES):
    print(f'  {name:<25}: {irf_proxy[0,j]:+.4f} pp')
print(f'\nNote: h=0 proxy responses reflect the estimated b vector;')
print(f'GDP growth and inflation can be non-zero (no recursive restriction).')

### Step 5 — Wild bootstrap confidence intervals

For the proxy SVAR we use a **wild bootstrap** with Mammen (1993) weights rather than the residual bootstrap used in Example 1. This is essential because the wild bootstrap preserves the contemporaneous correlation structure between the VAR residuals and the instrument — both are multiplied by the same scalar weight $w_t$, so $\text{Cov}(u_t^*, z_t^*) = w_t^2 \text{Cov}(u_t, z_t)$, leaving the identification ratio unaffected.

Mammen (1993) two-point distribution:
$$w_t = \begin{cases} -(\sqrt{5}-1)/2 & \text{with prob. } (\sqrt{5}+1)/(2\sqrt{5}) \\ (\sqrt{5}+1)/2 & \text{with prob. } (\sqrt{5}-1)/(2\sqrt{5}) \end{cases}$$
This ensures $\mathbb{E}[w_t] = 0$ and $\mathbb{E}[w_t^2] = 1$ while preserving the skewness correction.

In [ ]:
# ── Wild bootstrap for proxy SVAR ─────────────────────────────────────────────

def bootstrap_proxy(Y, z_full, p, H, n_boot, ci):
    """
    Wild bootstrap (Mammen 1993) for proxy SVAR confidence intervals.

    Key property: the SAME Mammen weight w_t multiplies both u_t and z_t,
    preserving Cov(u_t*, z_t*) = w_t^2 Cov(u_t, z_t).

    Returns bootstrap (median, lower, upper) at the specified CI level.
    Median is guaranteed to lie inside the bands.
    """
    T, K     = Y.shape
    _, B0, u0, _, X0 = estimate_var_ols(Y, p)
    T_eff    = len(u0)

    z_aligned = z_full[p:]         # align z to residual period
    n_use     = min(T_eff, len(z_aligned))
    z_trim    = z_aligned[:n_use]

    irfs_boot = np.zeros((n_boot, H+1, K))
    n_fail    = 0

    # Mammen two-point distribution parameters
    p_mammen  = (np.sqrt(5) + 1) / (2 * np.sqrt(5))
    w_low     = -(np.sqrt(5) - 1) / 2    # negative value (prob = p_mammen)
    w_high    =  (np.sqrt(5) + 1) / 2    # positive value (prob = 1 - p_mammen)

    for b in range(n_boot):
        try:
            # Draw T_eff scalar weights from Mammen distribution
            w = np.where(np.random.uniform(size=T_eff) < p_mammen, w_low, w_high)

            # Wild bootstrap residuals: multiply each row of U by its scalar weight
            u_star = u0 * w[:, np.newaxis]   # T_eff × K

            # Reconstruct Y* recursively from bootstrap residuals
            Y_star       = np.zeros_like(Y)
            Y_star[:p]   = Y[:p]     # actual initial conditions
            for t in range(T_eff):
                Y_star[p + t] = X0[t] @ B0 + u_star[t]

            # Re-estimate VAR on Y*
            _, B_b, u_b, _, _ = estimate_var_ols(Y_star, p)

            # Wild bootstrap instrument: SAME weights as residuals (key!)
            w_z   = w[:min(n_use, T_eff)]
            z_star = z_trim * w_z   # preserves Cov(u*, z*) = w² Cov(u, z)

            # Re-identify using bootstrap instrument
            b_b, _, _, _, _ = proxy_svar_id(u_b, z_star)
            irfs_boot[b]    = compute_irfs_proxy(B_b, b_b, p, K, H)
        except Exception:
            irfs_boot[b] = np.nan
            n_fail += 1

    if n_fail > 0:
        print(f'  Note: {n_fail} bootstrap replications failed (set to NaN)')

    alpha  = (1 - ci) / 2
    median = np.nanmedian(irfs_boot, axis=0)          # bootstrap median
    lower  = np.nanpercentile(irfs_boot, alpha*100, axis=0)   # lower band
    upper  = np.nanpercentile(irfs_boot, (1-alpha)*100, axis=0) # upper band
    return median, lower, upper


def bootstrap_cholesky(Y, p, H, n_boot, ci):
    """Residual bootstrap for Cholesky IRFs. Returns (median, lower, upper)."""
    T, K = Y.shape
    _, B0, u0, _, X0 = estimate_var_ols(Y, p)
    T_eff = len(u0)
    irfs_boot = np.zeros((n_boot, H+1, K))
    for b in range(n_boot):
        idx    = np.random.choice(T_eff, T_eff, replace=True)
        u_boot = u0[idx]
        Y_boot = np.zeros_like(Y)
        Y_boot[:p] = Y[:p]
        Y_boot[p:] = X0 @ B0 + u_boot
        try:
            _, B_b, _, Sig_b, _ = estimate_var_ols(Y_boot, p)
            P_b = cholesky(Sig_b, lower=True)
            IRF_b = compute_irfs_chol3(B_b, P_b, p, K, H)
            irfs_boot[b] = IRF_b[:, :, 2]   # MP shock column
        except Exception:
            irfs_boot[b] = np.nan
    alpha  = (1 - ci) / 2
    median = np.nanmedian(irfs_boot, axis=0)
    lower  = np.nanpercentile(irfs_boot, alpha*100, axis=0)
    upper  = np.nanpercentile(irfs_boot, (1-alpha)*100, axis=0)
    return median, lower, upper


print(f'Wild bootstrap (proxy SVAR): {E2_N_BOOT_PROXY} reps, {int(E2_CI_LEVEL*100)}% CI...')
med_p, lo_p, hi_p = bootstrap_proxy(
    Y2, data2['rr_shock'].values, E2_P, E2_H, E2_N_BOOT_PROXY, E2_CI_LEVEL)

print(f'Residual bootstrap (Cholesky): {E2_N_BOOT_CHOL} reps, {int(E2_CI_LEVEL*100)}% CI...')
med_c, lo_c, hi_c = bootstrap_cholesky(Y2, E2_P, E2_H, E2_N_BOOT_CHOL, E2_CI_LEVEL)

print('Done!')

### Step 6 — Figures

In [ ]:
# ── Helper: smooth bootstrap median ──────────────────────────────────────────
# Bootstrap medians can be noisy due to discrete jumps; a 5-quarter moving
# average smooths this while preserving the qualitative shape.
# The clip() ensures the smoothed median stays inside the confidence bands.

def smooth_median(med, lo, hi, window=5):
    med_s = uniform_filter1d(med, size=window)   # moving average
    med_s[0] = med[0]; med_s[1] = med[1]         # preserve edge values
    med_s[-1] = med[-1]; med_s[-2] = med[-2]
    return np.clip(med_s, lo, hi)                 # guarantee inside bands

In [ ]:
# ── Figure E2-1: Proxy SVAR IRFs ──────────────────────────────────────────────
h_axis2 = np.arange(E2_H + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle(
    f'Figure E2-1: Impulse Responses to Monetary Policy Shock\n'
    f'Proxy SVAR — Romer-Romer Instrument  |  {int(E2_CI_LEVEL*100)}% Wild Bootstrap CI  |  '
    f'{e2_first_q}–{e2_last_q}',
    fontsize=12, fontweight='bold'
)

for i, (ax, label) in enumerate(zip(axes, E2_VAR_NAMES)):
    med_s = smooth_median(med_p[:, i], lo_p[:, i], hi_p[:, i])
    ax.fill_between(h_axis2, lo_p[:, i], hi_p[:, i],
                    alpha=0.22, color=COLORS['proxy'],
                    label=f'{int(E2_CI_LEVEL*100)}% CI')
    ax.plot(h_axis2, med_s, color=COLORS['proxy'], linewidth=2.5,
            label='Bootstrap median')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Quarters after shock')
    ax.set_ylabel('Percentage points')
    ax.set_title(f'Response of {label}')
    ax.set_xlim(0, E2_H)
    ax.set_xticks([0, 8, 16, 24, 32, 40])
    ax.legend(fontsize=8)

fig.tight_layout()
path = os.path.join(OUT_DIR, 'e2_fig1_proxy_irfs.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show(); print(f'Saved: {path}')

In [ ]:
# ── Figure E2-2: Proxy SVAR vs Cholesky comparison ───────────────────────────
# This is the key diagnostic figure: if both methods agree qualitatively,
# we have confidence in the results. Large discrepancies suggest that the
# Cholesky recursive ordering is distorting the identified shock.
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle(
    f'Figure E2-2: Proxy SVAR vs Cholesky  |  {int(E2_CI_LEVEL*100)}% Bootstrap CI  |  '
    f'{e2_first_q}–{e2_last_q}',
    fontsize=12, fontweight='bold'
)

for i, (ax, label) in enumerate(zip(axes, E2_VAR_NAMES)):
    # Proxy SVAR: shaded band + smoothed median
    med_s = smooth_median(med_p[:, i], lo_p[:, i], hi_p[:, i])
    ax.fill_between(h_axis2, lo_p[:, i], hi_p[:, i],
                    alpha=0.18, color=COLORS['proxy'])
    ax.plot(h_axis2, med_s, color=COLORS['proxy'], linewidth=2.5,
            label='Proxy SVAR (R-R)')

    # Cholesky: smoothed median only (dashed line) — no band to avoid clutter
    med_cs = smooth_median(med_c[:, i], lo_c[:, i], hi_c[:, i])
    ax.plot(h_axis2, med_cs, color=COLORS['cholesky'], linewidth=2.5,
            linestyle='--', label='Cholesky')

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Quarters after shock')
    ax.set_ylabel('Percentage points')
    ax.set_title(f'Response of {label}')
    ax.set_xlim(0, E2_H)
    ax.set_xticks([0, 8, 16, 24, 32, 40])
    ax.legend(fontsize=9)

fig.tight_layout()
path = os.path.join(OUT_DIR, 'e2_fig2_comparison.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show(); print(f'Saved: {path}')

**What to look for in the comparison:** The inflation panel is the most informative. If Cholesky shows the price puzzle (inflation positive in early quarters) but proxy SVAR does not, this confirms that the recursive ordering imposes a distorting constraint: the Fed funds rate cannot respond to inflation within the quarter, forcing the Cholesky shock to absorb some anticipated inflation variation. The proxy SVAR avoids this by using the narrative instrument to directly isolate the exogenous policy component.

---
## Summary: Cholesky vs Proxy SVAR

In [ ]:
# ── Comparison table: impact effects and first-stage diagnostics ──────────────
print('=' * 70)
print('CHAPTER 4 SVAR IDENTIFICATION — SUMMARY')
print('=' * 70)

print('\n--- Example 1: 4-Variable Cholesky SVAR ---')
print(f'Sample: {first_q} – {last_q}  |  VAR({E1_P})  |  Bootstrap: {E1_N_BOOT} reps')
print(f'MP shock size (P[3,3]): {P_chol[3,3]:.4f} pp')
print(f'\nImpact effects (h=0) of MP shock:')
for i, name in enumerate(E1_VAR_NAMES):
    print(f'  {name:<22}: {irf_mp[0,i]:+.4f} pp')

print('\n--- Example 2: Proxy SVAR with Romer-Romer Instrument ---')
print(f'Sample: {e2_first_q} – {e2_last_q}  |  VAR({E2_P})')
print(f'Instrument: Romer-Romer (2004), updated by Wieland-Yang (2020)')
print(f'First-stage F-stat: {F_stat:.1f}  (> 10 = strong)  |  R² = {R2:.4f}')

print(f'\nImpact multipliers comparison (h=0):')
print(f'{"Variable":<25}  {"Cholesky":>10}  {"Proxy SVAR":>12}')
print('-' * 52)
for i, label in enumerate(E2_VAR_NAMES):
    print(f'{label:<25}  {irf_chol_mp[0,i]:>10.4f}  {irf_proxy[0,i]:>12.4f}')

print('\n--- Price Puzzle Check (inflation positive after contractionary shock) ---')
puz_chol  = np.where(irf_chol_mp[:20,1] > 0)[0]
puz_proxy = np.where(irf_proxy[:20,1] > 0)[0]
print(f'Cholesky: inflation > 0 at quarters {list(puz_chol) if len(puz_chol) else "none"}')
print(f'Proxy SVAR: inflation > 0 at quarters {list(puz_proxy) if len(puz_proxy) else "none"}')
print('(Fewer price puzzle quarters in proxy SVAR = better identification)')

---
## Exercises

**Exercise 4.1 (Cholesky ordering sensitivity):** In Example 1, change the ordering to $(i, \pi, Q, Y)$ — put the Fed funds rate *first* and GDP growth *last*. Re-run the Cholesky identification and compare the GDP and inflation IRFs. Does the price puzzle appear or disappear? This illustrates that Cholesky results can be sensitive to ordering choices.

**Exercise 4.2 (3-variable vs 4-variable):** Remove oil prices from Example 1 and re-run with just $(Y, \pi, i)$. Compare the inflation IRF to the 4-variable version. Is the price puzzle worse without oil? This replicates the classic Kim & Roubini (2000) result motivating commodity price augmentation.

**Exercise 4.3 (Alternative instrument):** Replace `resid_romer` with `resid_full` (the alternative Wieland-Yang column) in Example 2. Do the IRFs change substantially? A well-identified structural shock should be robust to reasonable perturbations of the instrument.

**Exercise 4.4 (Instrument validity):** Run a simple correlation test: `np.corrcoef(u2_gdp, z)` and `np.corrcoef(u2_inf, z)` where `u2_gdp` and `u2_inf` are the GDP and inflation VAR residuals. Under the proxy SVAR exogeneity assumption, these correlations should be close to zero. If they are large, the exclusion restriction is questionable.

**Exercise 4.5 (Extended sample):** Change `E2_SAMPLE_END` to `'2019-12-31'`. Does the first-stage F-statistic change? Are the proxy SVAR IRFs qualitatively similar? The Romer-Romer shocks are only available through 2007 — so extending the sample adds observations where $z_t$ is missing (zero-filled in the instrument file).

**Exercise 4.6 (Historical decomposition interpretation):** Using Figure E1-4, identify the three periods where the MP shock (orange) made the largest positive contribution to inflation. Do they correspond to known episodes of inflationary monetary policy (late 1960s, early 1970s)? Do they correspond to Volcker's aggressive tightening showing up as a large negative contribution?

---
*Macroeconometrics* | Alessia Paccagnini  
Companion code — Chapter 4: SVAR Identification | Last updated: March 2026  
Data: FRED-QD (2026-01), GDPC1, GDPDEF, FEDFUNDS, Romer-Romer (2004) / Wieland-Yang (2020)